# Phân tích khám phá dữ liệu gốc (Raw EDA)
## Santander Product Recommendation — Kaggle

| | |
|---|---|
| **Mục tiêu** | Hiểu cấu trúc, chất lượng và diễn biến theo thời gian của dữ liệu thô (~13,6 triệu dòng) trước khi làm sạch và thu gọn thành bộ 50K khách hàng |
| **Dataset** | [Santander Product Recommendation](https://www.kaggle.com/c/santander-product-recommendation/data) — `train_ver2.csv`, `test_ver2.csv` |
| **Phạm vi** | Train: 01/2015 – 05/2016 (17 tháng) · Test: 06/2016 (1 tháng) |
| **Đầu ra** | Danh sách vấn đề chất lượng dữ liệu và checklist tiền xử lý (Mục 8) |

### Mục lục

| # | Nội dung |
|---|---|
| 0 | Cài đặt môi trường |
| 1 | Tải dữ liệu & đổi tên cột |
| 2 | Tổng quan dữ liệu thô |
| 3 | Missing values — các khối tách biệt về nguồn gốc |
| 4 | Sản phẩm: tỷ lệ sở hữu & tương quan |
| 5 | Diễn biến theo thời gian (`fecha_dato`) |
| 6 | Age · Seniority · Household income |
| 7 | Cardinality & hướng encode |
| 8 | Tổng kết & checklist tiền xử lý |

---
## 0. Cài đặt môi trường

In [ ]:
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import missingno as msno
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': '#FAFAFA',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PALETTE = {
    'primary': '#E74C3C', 'secondary': '#3498DB', 'accent': '#2ECC71',
    'warm': '#E67E22', 'purple': '#9B59B6', 'teal': '#1ABC9C',
    'gold': '#F39C12', 'pink': '#E91E63', 'cyan': '#00BCD4',
    'dark': '#2C3E50', 'gray': '#95A5A6',
}

TEST_MONTH = pd.Timestamp('2016-06-28')   # tháng duy nhất của test_ver2.csv


def by_month(s):
    """Đổi index Timestamp -> 'YYYY-MM' để bảng số liệu dễ đọc."""
    s = s.copy()
    s.index = s.index.strftime('%Y-%m')
    return s

In [ ]:
INPUT_DIR = '/kaggle/input/competitions/santander-product-recommendation/'
WORK_DIR = '/kaggle/working/'

# Giải nén cả train và test
for fname in ['train_ver2.csv.zip', 'test_ver2.csv.zip']:
    with zipfile.ZipFile(INPUT_DIR + fname, 'r') as zf:
        zf.extractall(WORK_DIR)

---
## 1. Tải dữ liệu & đổi tên cột

Tên cột gốc bằng tiếng Tây Ban Nha nên được đổi sang tiếng Anh cho dễ đọc. Một số cột số trong file gốc bị lưu dạng chuỗi (có khoảng trắng) nên được ép về numeric, sau đó downcast để giảm RAM (train ≈ 1,7 GB sau khi tối ưu).

### 1.1 Từ điển tên cột

In [ ]:
# ── Ánh xạ tên cột Tây Ban Nha → Tiếng Anh ──────────────────────────────
rename_dict = {
    # Định danh & thời gian
    'fecha_dato': 'record_date',
    'ncodpers': 'customer_id',
    # Hồ sơ khách hàng
    'ind_empleado': 'employee_status',
    'pais_residencia': 'country_residence',
    'sexo': 'gender',
    'age': 'age',
    'fecha_alta': 'account_open_date',
    'ind_nuevo': 'is_new_customer',
    'antiguedad': 'seniority_months',
    'indrel': 'primary_customer_flag',
    'ult_fec_cli_1t': 'last_primary_date',
    'indrel_1mes': 'customer_type_start_month',
    'tiprel_1mes': 'relation_type_start_month',
    'indresi': 'is_resident',
    'indext': 'is_foreigner',
    'conyuemp': 'is_employee_spouse',
    'canal_entrada': 'acquisition_channel',
    'indfall': 'is_deceased',
    'tipodom': 'address_type',
    'cod_prov': 'province_code',
    'nomprov': 'province_name',
    'ind_actividad_cliente': 'is_active',
    'renta': 'household_income',
    'segmento': 'customer_segment',
    # 24 sản phẩm (nhị phân)
    'ind_ahor_fin_ult1': 'prod_saving_account',
    'ind_aval_fin_ult1': 'prod_guarantees',
    'ind_cco_fin_ult1': 'prod_current_account',
    'ind_cder_fin_ult1': 'prod_derivada_account',
    'ind_cno_fin_ult1': 'prod_payroll_account',
    'ind_ctju_fin_ult1': 'prod_junior_account',
    'ind_ctma_fin_ult1': 'prod_mas_particular_account',
    'ind_ctop_fin_ult1': 'prod_particular_account',
    'ind_ctpp_fin_ult1': 'prod_particular_plus_account',
    'ind_deco_fin_ult1': 'prod_short_term_deposit',
    'ind_deme_fin_ult1': 'prod_medium_term_deposit',
    'ind_dela_fin_ult1': 'prod_long_term_deposit',
    'ind_ecue_fin_ult1': 'prod_eaccount',
    'ind_fond_fin_ult1': 'prod_funds',
    'ind_hip_fin_ult1': 'prod_mortgage',
    'ind_plan_fin_ult1': 'prod_pension_plan',
    'ind_pres_fin_ult1': 'prod_loans',
    'ind_reca_fin_ult1': 'prod_taxes',
    'ind_tjcr_fin_ult1': 'prod_credit_card',
    'ind_valo_fin_ult1': 'prod_securities',
    'ind_viv_fin_ult1': 'prod_home_account',
    'ind_nomina_ult1': 'prod_payroll',
    'ind_nom_pens_ult1': 'prod_pension_payroll',
    'ind_recibo_ult1': 'prod_direct_debit',
}
INV_RENAME = {v: k for k, v in rename_dict.items()}

PRODUCT_COLS = [v for v in rename_dict.values() if v.startswith('prod_')]

PRODUCT_LABELS = {
    'prod_saving_account': 'Saving Account',
    'prod_guarantees': 'Guarantees',
    'prod_current_account': 'Current Account',
    'prod_derivada_account': 'Derivada Account',
    'prod_payroll_account': 'Payroll Account',
    'prod_junior_account': 'Junior Account',
    'prod_mas_particular_account': 'Mas Particular',
    'prod_particular_account': 'Particular Account',
    'prod_particular_plus_account': 'Particular Plus',
    'prod_short_term_deposit': 'Short Deposit',
    'prod_medium_term_deposit': 'Medium Deposit',
    'prod_long_term_deposit': 'Long Deposit',
    'prod_eaccount': 'E-Account',
    'prod_funds': 'Funds',
    'prod_mortgage': 'Mortgage',
    'prod_pension_plan': 'Pension Plan',
    'prod_loans': 'Loans',
    'prod_taxes': 'Taxes',
    'prod_credit_card': 'Credit Card',
    'prod_securities': 'Securities',
    'prod_home_account': 'Home Account',
    'prod_payroll': 'Payroll',
    'prod_pension_payroll': 'Pension Payroll',
    'prod_direct_debit': 'Direct Debit',
}

# Cột categorical — dùng TÊN GỐC vì khai báo dtype lúc đọc CSV (trước khi rename)
CATEGORICAL_COLS = [
    'ind_empleado', 'pais_residencia', 'sexo', 'indrel_1mes',
    'tiprel_1mes', 'indresi', 'indext', 'conyuemp', 'canal_entrada',
    'indfall', 'nomprov', 'segmento',
]

# Cột số bị lưu dạng chuỗi trong file gốc (test không có PRODUCT_COLS nên tự bỏ qua)
FORCE_NUMERIC = [
    'age', 'seniority_months', 'household_income', 'is_new_customer',
    'primary_customer_flag', 'is_active', 'address_type', 'province_code',
] + PRODUCT_COLS

print(f'Đã ánh xạ {len(rename_dict)} cột | {len(PRODUCT_COLS)} cột sản phẩm')

### 1.2 Hàm đọc dữ liệu & tải train

In [ ]:
def load_santander(path):
    """Đọc CSV Santander: parse ngày, khai báo categorical, đổi tên cột, ép numeric, downcast."""
    frame = pd.read_csv(
        path,
        parse_dates=['fecha_dato', 'fecha_alta'],
        dtype={c: 'category' for c in CATEGORICAL_COLS},
        low_memory=False,
    ).rename(columns=rename_dict)

    for col in FORCE_NUMERIC:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')

    for kind, target in [('int', 'integer'), ('float', 'float')]:
        for col in frame.select_dtypes(include=kind).columns:
            frame[col] = pd.to_numeric(frame[col], downcast=target)
    return frame


def report(frame, name):
    mem_mb = frame.memory_usage(deep=True).sum() / 1024**2
    print(f'[{name}] {frame.shape[0]:,} dòng × {frame.shape[1]} cột | RAM {mem_mb:,.1f} MB')
    print('         dtypes:', frame.dtypes.astype(str).value_counts().to_dict())

In [ ]:
%%time
df = load_santander(WORK_DIR + 'train_ver2.csv')
report(df, 'train')

In [ ]:
df.head()

### 1.3 Tải test

In [ ]:
%%time
df_test = load_santander(WORK_DIR + 'test_ver2.csv')
report(df_test, 'test')

In [ ]:
df_test.head()

---
## 2. Tổng quan dữ liệu thô

### 2.1 Quy mô & cấu trúc

In [ ]:
def panel_summary(frame):
    months = pd.to_datetime(np.sort(frame['record_date'].unique()))
    n_prod = sum(c in frame.columns for c in PRODUCT_COLS)
    return {
        'Số dòng': f'{len(frame):,}',
        'Khách hàng duy nhất': f'{frame["customer_id"].nunique():,}',
        'Số tháng': str(len(months)),
        'Khoảng thời gian': f'{months[0]:%Y-%m} → {months[-1]:%Y-%m}',
        'Dòng trùng (customer_id, record_date)': f'{frame.duplicated(subset=["customer_id", "record_date"]).sum():,}',
        'Tổng số cột': str(frame.shape[1]),
        '   · Feature': str(frame.shape[1] - n_prod - 2),
        '   · Sản phẩm': str(n_prod),
        '   · ID / Date': '2',
    }


pd.DataFrame({'Train': panel_summary(df), 'Test': panel_summary(df_test)})

### 2.2 Bảng tổng quan cột — Train

Sắp theo % missing giảm dần. Màu nền theo loại cột: xanh lá = sản phẩm · xanh dương = categorical · cam = numerical · tím = date · xám = ID.

In [ ]:
TYPE_COLORS = {
    'Product (binary)': '#27AE60', 'Categorical': '#3498DB',
    'Numerical': '#E67E22', 'ID / Date': '#95A5A6', 'Date': '#9B59B6',
}


def column_overview(frame, name):
    """Bảng tổng quan từng cột: loại, dtype, số giá trị unique, missing."""
    rows = []
    for col in frame.columns:
        if col in PRODUCT_COLS:
            ctype = 'Product (binary)'
        elif col in ('record_date', 'customer_id'):
            ctype = 'ID / Date'
        elif col == 'account_open_date':
            ctype = 'Date'
        elif frame[col].dtype.name in ('category', 'object'):
            ctype = 'Categorical'
        else:
            ctype = 'Numerical'
        rows.append({
            'Column': col,
            'Original': INV_RENAME.get(col, col),
            'Type': ctype,
            'Dtype': str(frame[col].dtype),
            'Unique': frame[col].nunique(),
            'Missing': frame[col].isnull().sum(),
            'Missing %': frame[col].isnull().mean() * 100,
        })

    ov = pd.DataFrame(rows).sort_values('Missing %', ascending=False).reset_index(drop=True)
    return (ov.style
            .format({'Missing': '{:,}', 'Missing %': '{:.2f}%', 'Unique': '{:,}'})
            .apply(lambda row: [f'background-color: {TYPE_COLORS.get(row["Type"], "")}22'] * len(row), axis=1)
            .background_gradient(subset=['Missing %'], cmap='Reds', vmin=0, vmax=100)
            .set_properties(**{'text-align': 'center'})
            .set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#2C3E50'), ('color', 'white'),
                                             ('text-align', 'center'), ('padding', '8px')]},
                {'selector': 'td', 'props': [('padding', '5px'), ('font-size', '10px')]},
            ])
            .set_caption(f'Tổng quan {frame.shape[1]} cột — {name} ({frame.shape[0]:,} dòng)'))


column_overview(df, 'Train (Kaggle gốc)')

### 2.3 Bảng tổng quan cột — Test

In [ ]:
column_overview(df_test, 'Test (Kaggle gốc)')

### 2.4 Nhận xét sơ bộ

**Missing theo khối** (kiểm chứng chi tiết ở Mục 3):

| Khối | Cột | Dòng thiếu | % |
|---|---|---:|---:|
| Profile | `gender`\*, `age`, `seniority_months`, `country_residence`, `employee_status`, `account_open_date`, `is_new_customer`, `is_foreigner`, `is_resident`, `is_deceased`, `is_active`, `primary_customer_flag` | 27,734 | 0.20% |
| Địa chỉ | `address_type` | 27,735 | 0.20% |
| Tỉnh | `province_code`, `province_name` | 93,591 | 0.69% |
| Quan hệ đầu kỳ | `customer_type_start_month`, `relation_type_start_month` | 149,781 | 1.10% |
| Segment / kênh | `customer_segment` / `acquisition_channel` | 189,368 / 186,126 | 1.39% / 1.36% |
| Payroll | `prod_payroll`, `prod_pension_payroll` | 16,063 | 0.12% |
| Thu nhập | `household_income` | 2,794,375 | 20.48% |
| Gần như trống | `last_primary_date` / `is_employee_spouse` | 13,622,516 / 13,645,501 | 99.82% / 99.99% |

\* `gender` thiếu 27,804 dòng — nhiều hơn khối profile đúng 70 dòng.

**Các điểm cần lưu ý**

- **Gần như vô dụng — nên loại thay vì impute**: `address_type` (chỉ 1 giá trị duy nhất trên toàn bộ 13,6 triệu dòng), `is_employee_spouse` (chỉ ~1,808 dòng có giá trị), `last_primary_date` (nếu giữ thì chỉ ở dạng flag có/không có ngày).
- **Cần kiểm tra outlier**: `age` có 120 giá trị unique — cao bất thường cho khách hàng cá nhân (Mục 6.2).
- **Cardinality cao**: `country_residence` (118) và `acquisition_channel` (162) — cần gộp nhóm hiếm trước khi encode (Mục 7).
- **`household_income`** thiếu 20.48% và không ngẫu nhiên (Mục 6.3).

---
## 3. Missing values — các khối tách biệt về nguồn gốc

Missing trong bộ dữ liệu này **không rải ngẫu nhiên** mà tập trung thành các khối. Mục này nhận diện khối bằng ma trận nullity (3.1), rồi kiểm chứng nguồn gốc từng khối (3.2 – 3.4) và tổng hợp (3.5).

### 3.1 Ma trận nullity

Một cụm 11 cột có số dòng thiếu gần trùng nhau (27,734 – 27,735 dòng ≈ 0,20%). Câu hỏi: đó có phải cùng một tập dòng không? Đếm số cột bị thiếu trên mỗi dòng và vẽ ma trận tương quan nullity, kèm `household_income`, `customer_segment`, `acquisition_channel` để so sánh.

In [ ]:
PROFILE_COLS = ['gender', 'age', 'seniority_months', 'account_open_date', 'employee_status',
                'is_resident', 'is_deceased', 'is_foreigner', 'is_new_customer', 'is_active']
cols_02pct = ['province_code'] + PROFILE_COLS          # 11 cột có ~0,20% missing

null_mask = df[cols_02pct].isnull()
overlap = null_mask.sum(axis=1)                        # số cột bị thiếu trên mỗi dòng (0 – 11)

print('Phân phối số cột bị thiếu trên mỗi dòng (trong 11 cột):')
display(overlap.value_counts().sort_index().rename('số dòng').to_frame().style.format('{:,}'))

msno.heatmap(df[cols_02pct + ['household_income', 'customer_segment', 'acquisition_channel']],
             figsize=(12, 8), fontsize=10)
plt.show()

**Kết quả** — missing tập trung thành các khối riêng biệt:

| Khối | Cột | Tương quan nullity | Kết luận |
|---|---|---|---|
| **1. Profile** | `gender, age, seniority_months, account_open_date, employee_status, is_resident, is_deceased, is_foreigner, is_new_customer, is_active` (10 cột) | ≈ 1 với nhau | Một sự kiện thiếu dữ liệu duy nhất, không phải 10 vấn đề độc lập |
| **`province_code`** | — | ≈ 0.5 với khối 1 | Pattern missing lệch một phần → xử lý riêng |
| **2. Segment / kênh** | `customer_segment`, `acquisition_channel` | ≈ 1 với nhau (sát 1); ≈ 0.4 với khối 1 | Cặp có chung nguồn, khác nguồn với khối profile |
| **`household_income`** | — | 0.1 – 0.2 với cả hai khối | Gần như độc lập, không thuộc khối nào |

**Hướng xử lý**: tạo 2 flag riêng `is_missing_profile` (khối 1) và `is_missing_segment_channel` (khối 2) thay vì gộp thành một flag chung để không mất thông tin; `province_code` và `household_income` xử lý độc lập.

### 3.2 Khối profile — 27,734 dòng thiếu đồng thời cả 11 cột

Giả thuyết ban đầu: đây là khách mới mở tài khoản, hồ sơ chưa hoàn chỉnh và sẽ được bổ sung sau. Kiểm tra bằng phân bố theo tháng và lịch sử xuất hiện của các khách này.

In [ ]:
mask_full = overlap == len(cols_02pct)                 # thiếu đồng thời cả 11 cột
df_full = df[mask_full]
ids_full = df_full['customer_id'].unique()

print(f'Dòng: {len(df_full):,} | Khách hàng duy nhất: {len(ids_full):,}')
display(by_month(df_full['record_date'].value_counts().sort_index()).rename('số dòng').to_frame().T)

# Các khách này có tháng khác đầy đủ thông tin không?
n_elsewhere = df.loc[df['customer_id'].isin(ids_full) & ~mask_full, 'customer_id'].nunique()
print(f'→ {n_elsewhere:,} / {len(ids_full):,} khách ({n_elsewhere / len(ids_full):.1%}) có tháng khác đầy đủ thông tin')

In [ ]:
# Các khách này xuất hiện bao nhiêu tháng trong toàn bộ dataset?
hist = df[df['customer_id'].isin(ids_full)].groupby('customer_id')['record_date'].agg(['count', 'min', 'max'])
last_month = df['record_date'].max()

display(hist['count'].describe().rename('số tháng xuất hiện').to_frame().T.style.format('{:.2f}'))
print(f'% còn xuất hiện ở tháng cuối của dataset ({last_month:%Y-%m}): {(hist["max"] == last_month).mean():.2%}')

**Kết quả** (27,734 dòng / 7,340 khách hàng)

- Chỉ xuất hiện trong **6 tháng đầu** (01 – 06/2015) và giảm đều: 6,953 → 5,940 → 5,091 → 4,292 → 3,597 → 1,861 dòng.
- Chỉ 309 / 7,340 khách (4.2%) có tháng khác đầy đủ thông tin → giả thuyết "hồ sơ sẽ được bổ sung sau" **không đúng** với đa số.
- Trung bình chỉ xuất hiện **4 tháng** (median 4, max 16/17) và chỉ **3.57%** còn xuất hiện ở tháng cuối (2016-05).

→ Bức tranh phù hợp với **nhóm khách rời dần / đóng tài khoản trong 6 tháng đầu 2015** (churn), không phải khách mới chờ bổ sung hồ sơ. Cân nhắc loại nhóm này khỏi tập train thay vì impute, hoặc dùng chính pattern này làm feature (`is_missing_profile`).

### 3.3 Giá trị lỗi trong `seniority_months`

`seniority_months` không thể âm. Đếm các giá trị âm, xem tác động lên mean rồi chuyển sang NaN.

In [ ]:
bad_seniority = df['seniority_months'] < 0
print(f'Dòng seniority âm: {bad_seniority.sum():,} | KH duy nhất: {df.loc[bad_seniority, "customer_id"].nunique():,}')
display(df.loc[bad_seniority, 'seniority_months'].value_counts().rename('số dòng').to_frame())

mean_before = df['seniority_months'].mean()
df.loc[bad_seniority, 'seniority_months'] = np.nan
df_test.loc[df_test['seniority_months'] < 0, 'seniority_months'] = np.nan   # xử lý tương tự cho test để hai tập nhất quán
print(f'mean(seniority_months): {mean_before:.2f} → {df["seniority_months"].mean():.2f} '
      f'| median: {df["seniority_months"].median():.0f}')

**Kết quả**: 38 dòng / 3 khách có `seniority_months = -999999` — mã lỗi ghi nhận cho cột `antiguedad`, không phải giá trị hợp lệ. Giá trị này kéo mean lệch xa median nếu không xử lý. Đã chuyển sang NaN; cần impute (hoặc loại) ở bước preprocess.

### 3.4 Khối `province_code` riêng

Giả thuyết ban đầu: khách mới nên chưa có tỉnh. Kiểm tra bằng `seniority_months` — so nhóm chỉ thiếu riêng `province_code` với toàn bộ dữ liệu (đã loại giá trị lỗi ở 3.3).

In [ ]:
mask_one = overlap == 1                                # dòng chỉ thiếu đúng 1 cột trong 11 cột
print('Cột nào bị thiếu khi chỉ có đúng 1 cột null?')
display(null_mask[mask_one].sum().rename('số dòng').to_frame().style.format('{:,}'))

df_prov = df[mask_one & df['province_code'].isnull()]
print(f'Dòng thiếu riêng province_code: {len(df_prov):,} | KH duy nhất: {df_prov["customer_id"].nunique():,}')

pd.concat({'Thiếu riêng province_code': df_prov['seniority_months'].describe(),
           'Toàn bộ dữ liệu': df['seniority_months'].describe()}, axis=1).style.format('{:,.1f}')

**Kết quả**: 65,857 dòng / 4,052 khách chỉ thiếu riêng `province_code` (93,591 dòng thiếu tỉnh = 65,857 + 27,734 dòng trùng khối profile). Nhóm này có seniority median **133 tháng (~11 năm)** so với **50 tháng (~4 năm)** của toàn bộ dữ liệu → đây là **khách lâu năm**, bác bỏ giả thuyết "khách mới". Gợi ý: tài khoản cũ/legacy chưa được cập nhật tỉnh cư trú, hơn là lỗi nhập liệu ngẫu nhiên.

### 3.5 Tổng hợp Mục 3

| Khối | Quy mô | Nguồn gốc (đã kiểm chứng) | Xử lý |
|---|---|---|---|
| Profile (10 cột) | 27,734 dòng / 7,340 KH | Churn dần trong 6 tháng đầu 2015 | Flag `is_missing_profile`; cân nhắc loại khỏi train |
| `province_code` riêng | 65,857 dòng / 4,052 KH | Khách lâu năm (seniority median 133 vs 50) | Impute riêng, không gộp vào flag profile |
| Segment + kênh | 189,368 / 186,126 dòng | Nguồn riêng, khác khối profile | Flag `is_missing_segment_channel` |
| `seniority_months = -999999` | 38 dòng / 3 KH | Mã lỗi | → NaN rồi impute |

Các việc chưa kiểm chứng của Mục 3 được liệt kê ở Mục 8.3.

---
## 4. Sản phẩm: tỷ lệ sở hữu & tương quan

### 4.1 Tỷ lệ sở hữu từng sản phẩm

In [ ]:
own_rate = df[PRODUCT_COLS].mean().sort_values() * 100   # % dòng (khách × tháng) đang sở hữu sản phẩm

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh([PRODUCT_LABELS[c] for c in own_rate.index], own_rate.values, color=PALETTE['secondary'])
for y, v in enumerate(own_rate.values):
    ax.text(v + 0.3, y, f'{v:.2f}%', va='center', fontsize=8)
ax.set_xlabel('% dòng (khách hàng × tháng) đang sở hữu sản phẩm')
ax.set_title('Tỷ lệ sở hữu 24 sản phẩm (train)')
ax.grid(False, axis='y')
plt.tight_layout()
plt.show()

**Nhận xét**: phân phối lệch mạnh giữa 24 sản phẩm — một số (`current_account`, `particular_account`, …) có tỷ lệ sở hữu cao, trong khi nhiều sản phẩm khác (`guarantees`, `home_account`, …) gần 0%. → Cảnh báo **class imbalance nặng** cho bài toán multi-label ở Checkpoint 2; cần chiến lược sampling/weighting riêng theo từng sản phẩm.

### 4.2 Tương quan giữa 24 sản phẩm

In [ ]:
corr = df[PRODUCT_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr.rename(index=PRODUCT_LABELS, columns=PRODUCT_LABELS),
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            square=True, linewidths=0.3, ax=ax)
ax.set_title('Tương quan giữa 24 sản phẩm')
ax.grid(False)
plt.tight_layout()
plt.show()

# Các cặp tương quan mạnh nhất (bỏ đường chéo)
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().sort_values(ascending=False)
top_pairs = pd.concat({'Dương (top 10)': pairs.head(10), 'Âm (top 5)': pairs.tail(5)}).rename('corr')
display(top_pairs.to_frame().style.format('{:.3f}'))

**Kết quả**

1. **Cụm "financially engaged" (~7 sản phẩm) xoay quanh `payroll_account`**: tương quan dương với `pension_payroll` (0.79), `payroll` (0.76), `direct_debit` (0.54), `credit_card` (0.39), `taxes` (0.32), `eaccount` (0.31), `particular_plus_account` (0.19). Các sản phẩm `payroll`, `credit_card`, `taxes`, `eaccount`, `direct_debit` cũng tương quan vừa với nhau (~0.2 – 0.5) ngay cả khi không qua `payroll_account` → nhóm khách dùng ngân hàng làm tài khoản chính (nhận lương/hưu, đóng thuế, thẻ tín dụng, ghi nợ tự động).
2. **`payroll` ↔ `pension_payroll` = 0.957** — cực cao, gần như trùng lặp; hiện tượng này đã được nhiều kernel Santander ghi nhận (kiểm tra kỹ ở 4.3).
3. **`current_account` tương quan âm** với cả cụm trên: −0.35 (`payroll_account`), −0.26 (`payroll`), −0.27 (`pension_payroll`) → gợi ý 2 phân khúc khác nhau: "basic" (giữ current_account, ít tương tác thêm) và "engaged" (chuyển hẳn sang payroll_account làm tài khoản chính).
4. **~14/24 sản phẩm gần như độc lập** (corr ≈ 0.00 – 0.05 với mọi sản phẩm khác): `guarantees`, `derivada_account`, `junior_account`, `mas_particular_account`, 3 loại deposit, `funds`, `mortgage`, `pension_plan`, `loans`, `home_account`, `securities` — mỗi sản phẩm phục vụ một nhu cầu riêng.

**Ý nghĩa cho modeling**: engineer `engagement_score` (đếm số sản phẩm trong cụm ~7) làm feature phụ cho các sản phẩm niche; thận trọng khi dùng `current_account` làm feature vì mang tín hiệu ngược chiều; với nhóm sản phẩm độc lập, không kỳ vọng feature chéo giúp nhiều.

### 4.3 `payroll` có phải là tập con của `pension_payroll`?

Cặp có tương quan 0.957 — kiểm tra xem hai cột trùng lặp hay có quan hệ phân cấp, và missing pattern của chúng có trùng nhau không.

In [ ]:
same_null = (df['prod_payroll'].isnull() == df['prod_pension_payroll'].isnull()).mean()
print(f'Tỷ lệ dòng có missing pattern trùng nhau: {same_null:.1%}')

pd.crosstab(df['prod_payroll'], df['prod_pension_payroll']).style.format('{:,}').set_caption(
    'prod_payroll (hàng) × prod_pension_payroll (cột)')

**Kết quả** — quan hệ **tập con**, không phải trùng lặp:

- `payroll = 1` ⟹ luôn `pension_payroll = 1` (0 ngoại lệ trong 745,961 dòng).
- Ngược lại, `pension_payroll = 1` xảy ra cả khi `payroll = 0` (64,124 dòng) — nhóm nhận lương hưu nhưng không qua sản phẩm payroll.
- Missing pattern trùng khớp **100%**: hai cột luôn null cùng nhau, cùng nguồn dữ liệu.

Diễn giải nghiệp vụ: `pension_payroll` có lẽ là "nhận khoản chi trả định kỳ (lương **hoặc** hưu)", còn `payroll` chỉ là tập con "nhận lương qua ngân hàng". Người đã về hưu có `pension_payroll = 1` mà `payroll = 0`.

**Kết luận cho modeling**

- Không phải multicollinearity cần loại bỏ (quan hệ phân cấp có ý nghĩa) → giữ cả hai và engineer thêm `is_pension_only = (pension_payroll == 1) & (payroll == 0)` để tách nhóm "chỉ nhận hưu, không đi làm".
- Vì missing cùng lúc 100%, xử lý chung bằng 1 flag missing cho cặp này khi impute.

---
## 5. Diễn biến theo thời gian (`fecha_dato`)

Dataset là panel data (mỗi khách × mỗi tháng), còn target của bài toán là **sản phẩm mới mua thêm** giữa các tháng — nên cần hiểu rõ cấu trúc theo thời gian trước khi chia train/validation.

### 5.1 Số khách hàng mỗi tháng

In [ ]:
monthly_customers = df.groupby('record_date')['customer_id'].nunique()

fig, ax = plt.subplots(figsize=(12, 4))
monthly_customers.plot(ax=ax, marker='o', color=PALETTE['secondary'])
ax.set_title('Số khách hàng theo tháng')
ax.set_ylabel('Số khách hàng')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v / 1e3:,.0f}K'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

display(by_month(monthly_customers).to_frame('Số khách hàng').T.style.format('{:,}'))

Số khách hàng tăng đột ngột ở tháng 07/2015 (632,110 → 829,817, +31%) — trong khi 6 tháng đầu gần như đi ngang.

### 5.2 Tỷ lệ sở hữu theo tháng (24 sản phẩm)

In [ ]:
monthly_prod_rate = df.groupby('record_date')[PRODUCT_COLS].mean().T    # hàng: sản phẩm, cột: tháng

fig, axes = plt.subplots(6, 4, figsize=(16, 14), sharex=True)
for ax, col in zip(axes.flat, PRODUCT_COLS):
    monthly_prod_rate.loc[col].plot(ax=ax, color=PALETTE['secondary'], marker='.')
    ax.set_title(PRODUCT_LABELS[col], fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)
fig.suptitle('Tỷ lệ sở hữu theo tháng', fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

### 5.3 Đặc trưng demographic theo tháng

Gộp train + test để nhìn liên tục qua 18 tháng; đường đứt đỏ đánh dấu tháng test (06/2016). Chỉ giữ các cột cần dùng để tiết kiệm RAM.

In [ ]:
COMBINED_COLS = ['customer_id', 'record_date', 'age', 'seniority_months', 'household_income',
                 'is_new_customer', 'is_active', 'is_resident', 'is_foreigner',
                 'primary_customer_flag', 'acquisition_channel', 'employee_status', 'customer_segment']
df_combined = pd.concat([df[COMBINED_COLS], df_test[COMBINED_COLS]], ignore_index=True)

demo_cols = ['age', 'seniority_months', 'household_income', 'is_new_customer',
             'is_active', 'is_resident', 'is_foreigner', 'primary_customer_flag']

demo = df_combined[['record_date'] + demo_cols].copy()
for col in ['is_resident', 'is_foreigner']:               # 'S' (Sí) -> 1, 'N' (No) -> 0, giữ NaN
    s = demo[col]
    demo[col] = (s == 'S').astype('float32').where(s.notna())
monthly_demo = demo.groupby('record_date')[demo_cols].mean()
del demo

fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharex=True)
for ax, col in zip(axes.flat, demo_cols):
    monthly_demo[col].plot(ax=ax, color=PALETTE['secondary'], marker='.')
    ax.axvline(TEST_MONTH, color=PALETTE['primary'], linestyle='--', alpha=0.5)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7, rotation=45)
fig.suptitle('Giá trị trung bình của feature demographic theo tháng', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Quan sát** (5.2 và 5.3 cùng chỉ về một hiện tượng)

- **5.2**: gần như toàn bộ 24 sản phẩm giảm tỷ lệ sở hữu **đột ngột tại cùng một thời điểm** giữa dataset — dạng bậc thang (step function), không phải giảm dần. Không sản phẩm nào tăng cùng lúc.
- **5.3**: đúng thời điểm đó, các feature demographic cũng đồng loạt "gãy": `age` giảm đột ngột; `household_income` tăng vọt rồi rớt xuống mức thấp ổn định; `is_active` giảm mạnh (~0.54 → 0.43); `is_foreigner` tăng vọt; `is_resident` tăng lên plateau; `is_new_customer` tăng dần suốt 2015.

**Giả thuyết**: một đợt khách mới ồ ạt gia nhập dataset (không phải khách hiện tại đổi hành vi). Nhóm này trẻ hơn, ít active hơn, nhiều foreigner hơn và gần như chưa sở hữu sản phẩm nào, nên **pha loãng** đồng loạt mọi tỷ lệ trung bình. Kiểm chứng ở 5.5 – 5.6.

### 5.4 Số lượt "Added" mỗi tháng

"Added" = sản phẩm chuyển từ 0 → 1 so với tháng trước của cùng khách — chính là target của bài toán.

In [ ]:
# Cần sort theo customer_id + record_date trước khi shift
df_sorted = df.sort_values(['customer_id', 'record_date'])
added_counts = pd.DataFrame(index=sorted(df['record_date'].unique())[1:])   # bỏ tháng đầu (không có lag)

for col in PRODUCT_COLS:
    lag = df_sorted.groupby('customer_id')[col].shift(1)
    added_counts[col] = ((df_sorted[col] == 1) & (lag == 0)).groupby(df_sorted['record_date']).sum()
del df_sorted

added_counts['total_added'] = added_counts.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
added_counts['total_added'].plot(ax=ax, marker='o', color=PALETTE['primary'])
ax.set_title('Tổng số lượt "Added" (mua thêm sản phẩm) theo tháng')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

display(by_month(added_counts['total_added']).to_frame('total_added').T.style.format('{:,}'))

**Kết quả**: tổng số lượt Added dao động khoảng 26.5K – 49K mỗi tháng và **không có đỉnh tại 07/2015** (33,713). Cực đại là 02/2016 (49,072), tiếp theo là 12/2015 (42,385) và 06/2015 (41,991).

Theo cách tính ở trên, tháng đầu tiên của mỗi khách không có `lag` nên **không bị đếm là Added** — vì vậy đợt khách mới 07/2015 không tự sinh ra Added giả trong chỉ số này. Đợt onboarding ảnh hưởng chủ yếu đến **mẫu số** (tỷ lệ sở hữu, feature demographic ở 5.2 – 5.3) chứ không thấy dấu hiệu làm phồng số lượt Added.

### 5.5 Khách hàng xuất hiện lần đầu theo tháng

In [ ]:
first_seen = df_combined.groupby('customer_id')['record_date'].min()
new_per_month = first_seen.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
new_per_month.iloc[1:].plot(ax=ax, marker='o', color=PALETTE['primary'])   # bỏ tháng đầu (baseline)
ax.set_title(f'Số khách hàng mới xuất hiện lần đầu theo tháng (bỏ 2015-01: {new_per_month.iloc[0]:,} — baseline)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v / 1e3:,.0f}K'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

display(by_month(new_per_month).to_frame('KH mới').T.style.format('{:,}'))

**Kết quả** — có **hai** hiện tượng tách biệt, không phải một:

| Giai đoạn | Số KH mới | Nhận xét |
|---|---:|---|
| 2015-01 | 625,457 | **Baseline**: tháng đầu của dataset nên mọi khách đều "mới" — artifact cấu trúc, không phải tăng trưởng thật |
| 2015-02 → 06 | 3,217 – 4,468 / tháng | Mức nền |
| **2015-07** | **193,115** | **Đột biến thật**: gấp ~43 – 60 lần mức nền; khớp đúng thời điểm "gãy" ở 5.2 – 5.3 |
| 2015-08 → 11 | 14,799 → 23,598 → 28,495 → 15,862 | Dư chấn, vẫn cao hơn nền nhiều lần |
| 2015-12 → 2016-05 | 7,667 → 4,693 | Hạ dần, vẫn cao hơn mức nền đầu 2015 (3 – 4K) |

### 5.6 Đợt 07/2015 đến từ đâu?

So sánh nhóm khách mới của 07/2015 (spike) với nhóm khách mới của 06/2015 (baseline) theo kênh thu nạp, trạng thái nhân viên và phân khúc.

In [ ]:
COHORTS = {'2015-07 (spike)': pd.Timestamp('2015-07-28'),
           '2015-06 (baseline)': pd.Timestamp('2015-06-28')}
cohort_dates = list(COHORTS.values())

rows = []
for label, month in COHORTS.items():
    ids = first_seen[first_seen == month].index
    sub = df_combined[df_combined['customer_id'].isin(ids) & df_combined['record_date'].isin(cohort_dates)]
    top3 = sub['acquisition_channel'].value_counts().head(3)
    rows.append({
        'Nhóm': label,
        'Số KH mới': f'{len(ids):,}',
        'Top 3 kênh (số dòng)': ', '.join(f'{k} ({v:,})' for k, v in top3.items()),
        '% employee_status = N': f'{(sub["employee_status"] == "N").mean():.2%}',
        '% segment = 02 - PARTICULARES': f'{(sub["customer_segment"] == "02 - PARTICULARES").mean():.1%}',
    })
pd.DataFrame(rows).set_index('Nhóm')

**Kết quả**

- Nhóm spike 07/2015 dồn vào 3 kênh **KFC (53,989), KAT (41,243), KHE (40,979)** — tổng ≈ 70% của 193K khách mới.
- Nhóm baseline 06/2015 dùng các kênh hoàn toàn khác: **KHM, KHN, KHK** — không trùng mã nào với nhóm spike.
- `employee_status` gần như toàn bộ là "N" ở cả hai nhóm → loại trừ giả thuyết onboarding nhân viên nội bộ; phân khúc lớn nhất là `02 - PARTICULARES` ở cả hai nhóm → không phải phân khúc đặc biệt.

**Diễn giải**: đợt tăng đến từ một nguồn thu nạp riêng, tách biệt hẳn với các kênh thông thường, và phù hợp với việc nhóm này chưa sở hữu sản phẩm nào khi vào hệ thống (pha loãng tỷ lệ ở 5.2). Giả thuyết cụ thể — mua lại/sáp nhập danh sách khách hàng từ nguồn bên ngoài — khớp với bằng chứng nhưng **chưa thể kiểm chứng trực tiếp** từ dữ liệu.

### 5.7 Tổng hợp Mục 5

1. **Baseline 2015-01** (625K khách "mới") là artifact của cấu trúc dataset, không phải tăng trưởng thật — cần loại khỏi mọi phân tích "khách hàng mới" dùng để ước lượng tốc độ tăng trưởng.
2. **Đợt đột biến 2015-07** (193,115 khách mới, ~43 – 60× mức nền) cùng dư chấn đến ~11/2015 là nguyên nhân trực tiếp của bậc thang đồng loạt ở tỷ lệ sở hữu 24 sản phẩm và các feature demographic.
3. **Rủi ro cho Checkpoint 2**: nếu chọn tháng train/validation nằm trong giai đoạn 07 – 11/2015 mà không xử lý, model có thể học nhầm pattern do distribution shift của đợt onboarding này. Hướng xử lý:
   - thêm feature nhóm `acquisition_channel` theo batch (KFC/KAT/KHE vs còn lại) và/hoặc cohort flag riêng cho 07/2015 (`is_new_customer` đã có nhưng chưa tách riêng đợt này);
   - loại hoặc weight riêng giai đoạn 07 – 11/2015 khi chia validation;
   - khi tách target Added/Dropped/Maintained, xử lý tường minh tháng đầu tiên của khách mới (không có lag).

---
## 6. Age · Seniority · Household income

### 6.1 Phân phối tổng quan

`household_income` được lọc `≤ P99` (không dùng `clip`) để histogram hiển thị đúng hình dạng phân phối — xem giải thích ở 6.3.

In [ ]:
income_p99 = df['household_income'].quantile(0.99)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['age'].hist(bins=50, ax=axes[0], color=PALETTE['secondary'])
axes[0].set_title('age')

df['seniority_months'].hist(bins=50, ax=axes[1], color=PALETTE['accent'])
axes[1].set_title('seniority_months')

df.loc[df['household_income'] <= income_p99, 'household_income'].hist(bins=60, ax=axes[2], color=PALETTE['warm'])
axes[2].set_title(f'household_income (≤ P99 = {income_p99:,.0f})')

plt.tight_layout()
plt.show()

df[['age', 'seniority_months', 'household_income']].describe().T.style.format('{:,.1f}')

### 6.2 Age — outlier

`age` có min = 2 và max = 164 — giá trị vô lý (bug nổi tiếng của dataset này). Đếm rồi xử lý: tuổi ≤ 18 gộp về 18, tuổi ≥ 100 chuyển thành NaN (ngưỡng có thể điều chỉnh).

In [ ]:
print(f'Age ≤ 18 : {(df["age"] <= 18).sum():,} dòng')
print(f'Age ≥ 100: {(df["age"] >= 100).sum():,} dòng')

df.loc[df['age'] <= 18, 'age'] = 18
df.loc[df['age'] >= 100, 'age'] = np.nan

### 6.3 Household income

**1. Hình dạng phân phối** — log-normal lệch phải điển hình: đỉnh quanh 100K, đuôi dài. Ở 6.1 dữ liệu được lọc `≤ P99` thay vì `clip(upper=P99)`, vì `clip` dồn mọi giá trị vượt ngưỡng vào đúng giá trị cap và tạo ra một cột giả ở mép phải — dễ bị hiểu nhầm là có nhóm thu nhập cao bất thường.

**2. Missing theo tỉnh** — kiểm tra tỷ lệ thiếu `household_income` theo `province_name`:

In [ ]:
income_miss_by_province = (df['household_income'].isnull()
                           .groupby(df['province_name'], observed=True).mean()
                           .sort_values(ascending=False))

(income_miss_by_province.head(10).rename('% thiếu household_income').to_frame()
 .style.format('{:.1%}').bar(color='#F5B7B1', vmin=0, vmax=1))

**Kết quả** — missing **không ngẫu nhiên**: 4 tỉnh đầu bảng thiếu ≈ 100% là Álava, Bizkaia, Gipuzkoa (xứ Basque) và Navarra. Nhiều khả năng đây là lý do cấu trúc chứ không phải lỗi thu thập ngẫu nhiên: các vùng này có chế độ tài khóa tự trị (Concierto/Convenio Económico), thu thuế qua cơ quan thuế vùng thay vì Agencia Tributaria trung ương — và `renta` có vẻ được lấy từ nguồn thuế trung ương nên không có dữ liệu cho 4 tỉnh này. Nhóm thứ hai (Baleares 44%; Lugo, Ourense, Ceuta, Tarragona, Pontevedra ≈ 29 – 33%) thấp hơn nhiều và có thể do nguyên nhân khác — chưa điều tra.

**Hướng xử lý**

- Impute bằng median toàn quốc sẽ làm lệch toàn bộ khách ở 4 tỉnh này (không phải missing at random).
- Tạo flag `income_missing_by_region` cho 4 tỉnh; phần missing còn lại impute theo median tỉnh/vùng.
- Với 4 tỉnh trên: hoặc bỏ income khỏi feature set cho nhóm này, hoặc impute theo median của `customer_segment` / `age_bucket` (vì tỉnh không mang tín hiệu).

### 6.4 Seniority & ngày mở tài khoản

Histogram `seniority_months` gộp cả 17 tháng có dạng răng cưa (sawtooth) — một phần do panel data (mỗi khách xuất hiện nhiều lần). Xem lại trên **một lát cắt duy nhất** (tháng cuối) cho sạch, rồi đối chiếu với `account_open_date`.

In [ ]:
snap = df.loc[df['record_date'] == df['record_date'].max()].copy()
snap['open_month'] = snap['account_open_date'].dt.to_period('M')
snap['open_year'] = snap['account_open_date'].dt.year
snap['open_calendar_month'] = snap['account_open_date'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(16, 4), gridspec_kw={'width_ratios': [1, 2]})
snap['seniority_months'].hist(bins=60, ax=axes[0], color=PALETTE['accent'])
axes[0].set_title(f'seniority_months tại {df["record_date"].max():%Y-%m}')

snap['open_month'].value_counts().sort_index().plot(ax=axes[1], color=PALETTE['accent'])
axes[1].set_title('Số tài khoản mở theo tháng (nhìn từ snapshot cuối)')
axes[1].tick_params(axis='x', rotation=90, labelsize=6)
plt.tight_layout()
plt.show()

# Tính chu kỳ theo tháng trong năm: số đếm thô vs chuẩn hóa theo năm
raw_by_month = snap.groupby('open_calendar_month').size()
pivot = snap.groupby(['open_year', 'open_calendar_month']).size().unstack(fill_value=0)
norm_by_month = pivot.div(pivot.sum(axis=1), axis=0).loc[2012:].mean()   # % trong năm đó → loại hiệu ứng quy mô giữa các năm

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
raw_by_month.plot(kind='bar', ax=axes[0], color=PALETTE['secondary'])
axes[0].set_title('Số tài khoản mở theo tháng trong năm (thô)')
(norm_by_month * 100).plot(kind='bar', ax=axes[1], color=PALETTE['secondary'])
axes[1].axhline(100 / 12, color=PALETTE['primary'], linestyle='--', alpha=0.6, label='Nếu đều: 8.3%/tháng')
axes[1].set_title('% tài khoản mở theo tháng (chuẩn hóa theo năm, từ 2012)')
axes[1].legend()
for ax in axes:
    ax.set_xlabel('Tháng trong năm')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**Kết quả**

- Số tài khoản mở mới **tăng dần theo năm** (tăng trưởng organic), rõ nhất từ 2011 trở đi.
- **Seasonal thật**: sau khi chuẩn hóa theo năm (loại hiệu ứng "năm gần đây đông hơn"), tháng 10 vẫn nổi bật nhất (~17.5%, gần gấp đôi mức đều 8.3%/tháng); Q3 – Q4 (tháng 7, 9, 10, 11) chiếm áp đảo, tháng 6 thấp nhất.
- Mùa cao điểm Q3 trùng thời điểm với đợt onboarding 07/2015 (Mục 5) — có thể đó là một phần của chiến dịch định kỳ hàng năm chứ không phải sự kiện đơn lẻ (giả thuyết, chưa kiểm chứng).

**Hướng xử lý**: engineer `open_month_of_year` dạng cyclical (sin/cos) để capture seasonal.

### 6.5 Liên hệ với số sản phẩm sở hữu

In [ ]:
n_prod = df[PRODUCT_COLS].sum(axis=1)
df['n_products'] = n_prod

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['age', 'seniority_months', 'household_income']):
    valid = df[[col, 'n_products']].dropna()
    sample = valid.sample(min(200_000, len(valid)), random_state=42)
    ax.scatter(sample[col], sample['n_products'], alpha=0.05, s=5, color=PALETTE['primary'])
    ax.set_xlabel(col)
    ax.set_ylabel('n_products')
    if col == 'household_income':
        ax.set_xlim(0, income_p99)
plt.tight_layout()
plt.show()

df[['age', 'seniority_months', 'household_income', 'n_products']].corr()['n_products'].drop('n_products') \
    .rename('corr với n_products').to_frame().style.format('{:.3f}')

In [ ]:
# Rõ hơn scatter: trung bình n_products theo nhóm tuổi
age_bins = [0, 18, 25, 35, 45, 55, 65, 100]
age_labels = ['≤18', '19–25', '26–35', '36–45', '46–55', '56–65', '66+']
age_avg = n_prod.groupby(pd.cut(df['age'], bins=age_bins, labels=age_labels), observed=True).mean()

fig, ax = plt.subplots(figsize=(8, 4))
age_avg.plot(kind='bar', ax=ax, color=PALETTE['secondary'])
for i, v in enumerate(age_avg.values):
    ax.text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=9)
ax.set_title('Số sản phẩm trung bình theo nhóm tuổi')
ax.set_xlabel('Nhóm tuổi')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**Kết quả**

| Biến | Corr với `n_products` | Nhận xét |
|---|---:|---|
| `seniority_months` | **0.31** | Mạnh nhất trong 3 biến — khách gắn bó lâu tích lũy nhiều sản phẩm hơn; khớp với nhận định nhóm "financially engaged" thường là khách lâu năm |
| `age` | 0.18 | Trung bình; quan hệ không đơn điệu (xem bên dưới) |
| `household_income` | **0.04** | Gần như không tương quan — trái với trực giác "thu nhập cao → nhiều sản phẩm" |

**`age` theo vòng đời** — dạng chuông rõ ràng: ≤18: 0.91 · 19–25: 0.90 · 26–35: 1.29 · 36–45: 1.83 · **46–55: 1.98** · 56–65: 1.78 · 66+: 1.47. Phù hợp với vòng đời tài chính điển hình: người trẻ mới bắt đầu sự nghiệp nên ít nhu cầu sản phẩm phức tạp (mortgage, pension_plan, đầu tư); tích lũy dần qua tuổi lao động chính, đạt đỉnh quanh 46 – 55 (thu nhập ổn định, có nhà, chuẩn bị hưu trí); giảm nhẹ sau 65 (có thể do đơn giản hóa danh mục hoặc đã tất toán một số sản phẩm như mortgage).

**`household_income`** gần như vô dụng để dự đoán *số lượng* sản phẩm, dù là cột tốn công xử lý nhất (missing 20.5%, lệch theo tỉnh). Không nên loại bỏ hoàn toàn: income có thể vẫn hữu ích cho dự đoán *loại* sản phẩm cụ thể (mortgage, pension_plan) — giả thuyết chưa kiểm chứng — chỉ đừng kỳ vọng nó giải thích được engagement chung. `age` (qua vòng đời) là yếu tố dự đoán tốt hơn income cho mức độ đa dạng sản phẩm.

---
## 7. Cardinality & hướng encode

Với cột cardinality cao, xem nhóm top-10 chiếm bao nhiêu % để biết mức độ long-tail.

In [ ]:
cat_cols = ['employee_status', 'country_residence', 'gender', 'is_resident', 'is_foreigner',
            'acquisition_channel', 'province_code', 'province_name', 'customer_segment',
            'customer_type_start_month', 'relation_type_start_month']

card = pd.DataFrame({
    'Cardinality': df[cat_cols].nunique(),
    'Top-10 chiếm (%)': pd.Series({c: df[c].value_counts(normalize=True).head(10).sum() * 100 for c in cat_cols}),
}).sort_values('Cardinality', ascending=False)
display(card.style.format({'Top-10 chiếm (%)': '{:.1f}'}))

print('country_residence — top 3 (%):')
display((df['country_residence'].value_counts(normalize=True).head(3) * 100).rename('%').to_frame().style.format('{:.2f}'))

print('customer_type_start_month — các category hiện có:', list(df['customer_type_start_month'].cat.categories))

**Kết quả**

| Cột | Cardinality | Đặc điểm | Hướng encode |
|---|---:|---|---|
| `acquisition_channel` | 162 | Long-tail nặng: top 10 chiếm 90.4%, 152 giá trị còn lại chỉ ~9.6% | Frequency/target encoding, gộp rare (< 0.1%) → `OTHER`. One-hot sẽ tạo 162 cột thưa |
| `province_code` / `province_name` | 52 | Trùng thông tin — chỉ giữ 1 trong 2; top 10 chiếm 67.3% | Target/frequency encoding (giữ được tín hiệu missing income theo tỉnh tốt hơn one-hot); one-hot 52 cột cũng khả thi |
| `country_residence` | 118 | Gần-constant: ES = 99.52% | Binary hóa `is_spain`, bỏ cột gốc |
| Các cột còn lại | ≤ 9 | Cardinality thấp | One-hot bình thường |

**Lưu ý**: `customer_type_start_month` báo 9 giá trị nhưng thực chất chỉ có 5 (`1`, `2`, `3`, `4`, `P`) — các cặp `'1'`/`'1.0'`, `'2'`/`'2.0'`, … là cùng một giá trị bị lưu khác định dạng chuỗi. Cần chuẩn hóa trước khi encode, nếu không sẽ bị tách thành các category riêng.

---
## 8. Tổng kết & checklist tiền xử lý

**Phạm vi phân tích**: `train_ver2.csv` (13,647,309 dòng × 48 cột, 956,645 khách hàng, 01/2015 – 05/2016) và `test_ver2.csv` (929,615 dòng × 24 cột, tháng 06/2016).

### 8.1 Các phát hiện chính

**1. Missing values — các khối tách biệt về nguồn gốc** (Mục 3)

- **Khối profile** (thiếu đồng thời 11 cột, gồm 10 cột profile + `province_code`; 27,734 dòng / 7,340 KH, ~0.20%): các khách này chỉ xuất hiện trung bình **4 tháng** (median) và chỉ **3.57%** còn tồn tại đến tháng cuối → nhóm **rời dần trong 6 tháng đầu 2015**, không phải "hồ sơ mới chờ hoàn thiện".
- **`province_code`** (93,591 dòng, ~0.69%): trong đó 65,857 dòng chỉ riêng tỉnh bị thiếu (phần còn lại trùng khối profile). Nhóm này có seniority median **133 tháng** vs **50 tháng** toàn population → **khách lâu năm/legacy**, không phải khách mới.
- **`customer_segment` / `acquisition_channel`** (189,368 và 186,126 dòng, ~1.4%): tương quan nullity ≈ 1 với nhau nhưng chỉ ~0.4 với khối profile → nguồn gốc riêng.
- **`customer_type_start_month` / `relation_type_start_month`** (149,781 dòng, ~1.1%): chưa xác định quan hệ với các khối trên (xem 8.3).
- **`household_income`** (missing 20.5%, ~2.79M dòng): **không ngẫu nhiên theo tỉnh** — 4 tỉnh (Álava, Bizkaia, Gipuzkoa, Navarra) thiếu gần 100%.
- **Lỗi dữ liệu**: `seniority_months = -999999` ở 38 dòng / 3 khách (mã lỗi cá biệt); `address_type` (constant) và `is_employee_spouse` (missing 99.99%) gần như vô dụng.

**2. Tỷ lệ sở hữu sản phẩm — imbalance nghiêm trọng** (Mục 4.1)

Phân phối lệch mạnh giữa 24 sản phẩm → cảnh báo **class imbalance nặng** cho bài toán multi-label ở Checkpoint 2; cần chiến lược sampling/weighting riêng theo từng sản phẩm.

**3. Tương quan giữa các sản phẩm — một cụm chính, phần lớn độc lập** (Mục 4.2 – 4.3)

- **Cụm "financially engaged"** (~7 sản phẩm) xoay quanh `payroll_account`: tương quan cao với `pension_payroll` (0.79), `payroll` (0.76), vừa phải với `direct_debit` (0.54), `credit_card` (0.39), `taxes` (0.32), `eaccount` (0.31).
- **`payroll` ⊆ `pension_payroll`** (corr 0.957): crosstab cho thấy quan hệ tập con hoàn toàn (payroll = 1 ⟹ pension_payroll = 1), missing trùng khớp 100% → giữ cả hai, thêm `is_pension_only`.
- **`current_account` tương quan âm** (−0.35 đến −0.26) với cả cụm trên → gợi ý 2 phân khúc "basic" vs "engaged".
- **~14/24 sản phẩm gần như độc lập** (corr < 0.05) — nên model riêng lẻ, không kỳ vọng feature chéo hỗ trợ nhiều.

**4. Diễn biến theo `fecha_dato` — đợt onboarding hàng loạt 07/2015** (Mục 5)

- Baseline 2015-01 (625K khách "mới") là artifact cấu trúc dataset, không phải tăng trưởng thật.
- **Đột biến 2015-07**: 193,115 khách mới (~ 43 – 60× mức nền 3.2K – 4.5K/tháng), dồn vào 3 kênh `KFC/KAT/KHE` (~ 70%), hoàn toàn khác kênh baseline (`KHM/KHN/KHK`); `employee_status` gần 100% "N" → đợt thu nạp từ một nguồn riêng, không phải nhân viên nội bộ (giả thuyết mua lại/sáp nhập danh sách khách chưa kiểm chứng được). Dư chấn kéo dài đến ~11/2015.
- Đợt này gây "gãy" đồng loạt tỷ lệ sở hữu 24 sản phẩm và các feature demographic do hiệu ứng pha loãng từ nhóm khách mới chưa có sản phẩm. Số lượt Added **không** có đỉnh ở 07/2015.

**5. Age / Seniority / Household income** (Mục 6)

- **`age`**: outlier (min = 2, max = 164). Theo vòng đời, số sản phẩm trung bình thấp ở 2 đầu (≤ 25 tuổi: ~0.9), **đỉnh ở nhóm 46 – 55 tuổi (1.98)**, giảm nhẹ sau 65 (1.47).
- **`seniority_months`**: corr với số sản phẩm sở hữu **mạnh nhất trong nhóm demographic (0.31)**. `account_open_date` tăng dần theo năm (rõ nhất từ 2011) và **seasonal rõ rệt** — cao điểm tháng 9 – 11 (đỉnh tháng 10, ~17.5%), thấp điểm tháng 6.
- **`household_income`**: corr với số sản phẩm sở hữu **gần như 0 (0.04)** dù tốn công xử lý nhất. Không loại bỏ hoàn toàn vì có thể vẫn hữu ích cho dự đoán *loại* sản phẩm cụ thể (mortgage, pension_plan).

**6. Cardinality & hướng encode** (Mục 7)

| Cột | Cardinality | Ghi chú | Hướng encode |
|---|---:|---|---|
| `acquisition_channel` | 162 | Long-tail nặng, top 10 chiếm 90.4% | Frequency/target encoding + gộp rare → `OTHER` |
| `province_code` | 52 | Top 10 chiếm 67.3% | Target/frequency encoding |
| `country_residence` | 118 | Gần-constant (ES = 99.52%) | Binary hóa `is_spain`, drop cột gốc |
| `customer_type_start_month`, `relation_type_start_month`, `employee_status`, `customer_segment`, `gender`, `is_resident`, `is_foreigner` | ≤ 9 (`customer_type_start_month` thực chất 5 sau chuẩn hóa) | Cardinality thấp | One-hot bình thường |

### 8.2 Checklist cho bước Preprocess (Checkpoint 1)

1. Chuyển `seniority_months = -999999` → NaN; xử lý `age`: ≤ 18 → 18, ≥ 100 → NaN (đúng theo cell ở Mục 6.2).
2. Tạo flag `is_missing_profile` (10 cột profile) và `is_missing_segment_channel`; impute `household_income` theo vùng/segment (không dùng median toàn quốc).
3. Drop `address_type`, `is_employee_spouse` (gần như vô dụng).
4. Chuẩn hóa `customer_type_start_month` (`'1.0'` → `'1'`, …) trước khi encode.
5. Encode: frequency/target cho `acquisition_channel` (kèm gộp rare) và `province_code`; binary `is_spain` cho `country_residence`; one-hot cho các cột cardinality thấp.
6. Engineer feature: `is_pension_only`, `open_month_of_year` (cyclical), `engagement_score` (đếm sản phẩm trong cụm payroll_account), cohort flag cho đợt onboarding 07/2015.
7. Khi tách target Added/Dropped/Maintained: xử lý tường minh tháng đầu tiên của khách mới (không có lag) và loại trừ ảnh hưởng của đợt onboarding hàng loạt khỏi tín hiệu "mua thêm" thật.

### 8.3 Việc còn mở

- Cặp `customer_type_start_month` / `relation_type_start_month` (149,781 dòng, 1.10%) chưa có trong ma trận nullity → chạy lại `msno.heatmap` có thêm 2 cột này để biết nó lồng vào khối province hay tách biệt hẳn.
- Kích thước các khối missing tăng dần (27,734 < 93,591 < 149,781 < ~187,000) gợi ý dạng "vòng tròn lồng nhau" — nên kiểm chứng bằng crosstab/overlap trước khi kết luận.
- Định lượng imbalance theo từng sản phẩm (Mục 4.1) để chọn chiến lược sampling/weighting cho Checkpoint 2.